In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.preprocessing import StandardScalerimport torchimport shap

# Load Single Edge MLP Models

In [ ]:
nn_fval_path = './output_data/model_FValue.pt'nn_w1_path = './output_data/model_W1.pt'nn_fval = torch.load(nn_fval_path)nn_w1 = torch.load(nn_w1_path)nn_fval.eval()nn_w1.eval()

# Load and Prepare Data

In [ ]:
test_filename = './training_data/data_bias_e5_1.csv'df = pd.read_csv(test_filename)input_feats = ['gamma1', 'lambda1', 'delta', 'epsilon', 'NRow', 'NCol']output_feat_fval = ['FValue']output_feat_w1 = ['W1']X = df[input_feats].valuesY_fval = df[output_feat_fval].valuesY_w1 = df[output_feat_w1].valuesPredictorScaler = StandardScaler()PredScaleFit = PredictorScaler.fit(X)X_scaled = PredScaleFit.transform(X)

# Shapley Analysis

In [ ]:
class NN_wrapper:    def __init__(self, model):        self.model = model    def predict(self, X):        X_tensor = torch.tensor(X, dtype=torch.float32)        y_preds_tensor = self.model(X_tensor)        y_preds = y_preds_tensor.detach().numpy()        return y_predsnn_fval_wrapper = NN_wrapper(nn_fval)nn_w1_wrapper = NN_wrapper(nn_w1)

In [ ]:
explainer_fval = shap.Explainer(nn_fval_wrapper.predict, shap.sample(X_scaled, 100))shap_values_fval = explainer_fval(X_scaled)explainer_w1 = shap.Explainer(nn_w1_wrapper.predict, shap.sample(X_scaled, 100))shap_values_w1 = explainer_w1(X_scaled)

# Beeswarm Plots

In [ ]:
shap.plots.beeswarm(shap_values_fval)

In [ ]:
shap.plots.beeswarm(shap_values_w1)

# Mean Absolute SHAP Values

In [ ]:
print('phi_tau SHAP values:')for i in range(6):    print(f'{input_feats[i]}: {np.mean(np.abs(shap_values_fval.values[:,i])):.3f}')print()print('Delta_tau SHAP values:')for i in range(6):    print(f'{input_feats[i]}: {np.mean(np.abs(shap_values_w1.values[:,i])):.3f}')